In [15]:
import pandas as pd
import numpy as np
from math import pi, acos, sqrt

In [16]:
df = pd.read_csv("Team and Court Geometry Data for NN.csv")
df.columns = df.columns.str.strip().str.lower()
df = df.loc[:, ~df.columns.str.contains("^unnamed", case=False)]

print(df.columns.tolist())
display(df.head())

['team', 'r_3pt_radius', 'court_width', 'baseline_width', 'hoop_to_baseline', 'corner3_distance', 'corner3_availability', 'arc_length_inbounds', 'area_inside_arc', 'area_outside_arc', 'twopa_per_game', 'threepa_per_game', 'corner3_pa', 'above_break3_pa', 'rim_attempt_share', 'midrange_share', 'free_throw_rate', 'eppg', 'epps', 'efg_expected', 'threepar', 'team_rim_rate', 'team_corner3_rate', 'off_reb_rate', 'pace', 'opp_3par_allowed', 'opp_rim_fg_allowed', 'switch_rate', 'turnover_rate', 'def_reb_rate']


,team,r_3pt_radius,court_width,baseline_width,hoop_to_baseline,corner3_distance,corner3_availability,arc_length_inbounds,area_inside_arc,area_outside_arc,...,threepar,team_rim_rate,team_corner3_rate,off_reb_rate,pace,opp_3par_allowed,opp_rim_fg_allowed,switch_rate,turnover_rate,def_reb_rate
0,Cavaliers2016,23.75,50,50,4,22,0,0,0,0,...,0.362,0.292,0.087,0.208,93.3,0.303,0.604,0.55,0.131,0.782
1,Warriors2016,23.75,50,50,4,22,0,0,0,0,...,0.416,0.285,0.095,0.229,99.3,0.287,0.582,0.72,0.149,0.789
2,Thunder2025,23.75,50,50,4,22,0,0,0,0,...,0.423,0.301,0.101,0.241,100.8,0.348,0.596,0.70,0.154,0.792
3,Pacers2025,23.75,50,50,4,22,0,0,0,0,...,0.398,0.287,0.094,0.226,101.5,0.356,0.603,0.62,0.139,0.774
4,Cavaliers2016,24.50,50,50,4,22,0,0,0,0,...,0.362,0.292,0.087,0.208,93.3,0.303,0.604,0.55,0.131,0.782


In [17]:
def compute_corner3_availability(r, court_width):
    half_width = court_width / 2.0
    if r <= half_width:
        return 1.0
    theta = acos(half_width / r)
    return max(0.0, 1.0 - (2 * theta) / pi)

def compute_arc_length_inbounds(r, court_width):
    half_width = court_width / 2.0
    if r <= half_width:
        return pi * r
    theta = acos(half_width / r)
    usable_angle = pi - 2 * theta
    return max(0.0, r * usable_angle)

def compute_area_inside_arc(r, court_width):
    half_width = court_width / 2.0
    if r <= half_width:
        return 0.5 * pi * r**2
    theta = acos(half_width / r)
    semicircle_area = 0.5 * pi * r**2
    segment_area = r**2 * theta - half_width * sqrt(r**2 - half_width**2)
    clipped_area = semicircle_area - segment_area
    return max(0.0, clipped_area)

def compute_area_outside_arc(court_width, hoop_to_baseline, area_inside_arc):
    offensive_half_length = 47.0 - hoop_to_baseline
    total_half_area = court_width * offensive_half_length
    return max(0.0, total_half_area - area_inside_arc)

In [18]:
df["corner3_availability"] = df.apply(
    lambda row: compute_corner3_availability(row["r_3pt_radius"], row["court_width"]),
    axis=1
)

df["arc_length_inbounds"] = df.apply(
    lambda row: compute_arc_length_inbounds(row["r_3pt_radius"], row["court_width"]),
    axis=1
)

df["area_inside_arc"] = df.apply(
    lambda row: compute_area_inside_arc(row["r_3pt_radius"], row["court_width"]),
    axis=1
)

df["area_outside_arc"] = df.apply(
    lambda row: compute_area_outside_arc(
        row["court_width"],
        row["hoop_to_baseline"],
        row["area_inside_arc"]
    ),
    axis=1
)

display(df.head())

,team,r_3pt_radius,court_width,baseline_width,hoop_to_baseline,corner3_distance,corner3_availability,arc_length_inbounds,area_inside_arc,area_outside_arc,...,threepar,team_rim_rate,team_corner3_rate,off_reb_rate,pace,opp_3par_allowed,opp_rim_fg_allowed,switch_rate,turnover_rate,def_reb_rate
0,Cavaliers2016,23.75,50,50,4,22,1.0,74.612826,886.027303,1263.972697,...,0.362,0.292,0.087,0.208,93.3,0.303,0.604,0.55,0.131,0.782
1,Warriors2016,23.75,50,50,4,22,1.0,74.612826,886.027303,1263.972697,...,0.416,0.285,0.095,0.229,99.3,0.287,0.582,0.72,0.149,0.789
2,Thunder2025,23.75,50,50,4,22,1.0,74.612826,886.027303,1263.972697,...,0.423,0.301,0.101,0.241,100.8,0.348,0.596,0.70,0.154,0.792
3,Pacers2025,23.75,50,50,4,22,1.0,74.612826,886.027303,1263.972697,...,0.398,0.287,0.094,0.226,101.5,0.356,0.603,0.62,0.139,0.774
4,Cavaliers2016,24.50,50,50,4,22,1.0,76.969020,942.870495,1207.129505,...,0.362,0.292,0.087,0.208,93.3,0.303,0.604,0.55,0.131,0.782


In [19]:
final_columns = [
    "team",
    "r_3pt_radius",
    "court_width",
    "baseline_width",
    "hoop_to_baseline",
    "corner3_distance",
    "corner3_availability",
    "arc_length_inbounds",
    "area_inside_arc",
    "area_outside_arc",
    "twopa_per_game",
    "threepa_per_game",
    "corner3_pa",
    "above_break3_pa",
    "rim_attempt_share",
    "midrange_share",
    "free_throw_rate",
    "eppg",
    "epps",
    "efg_expected",
    "threepar",
    "team_rim_rate",
    "team_corner3_rate",
    "off_reb_rate",
    "pace",
    "opp_3par_allowed",
    "opp_rim_fg_allowed",
    "switch_rate",
    "turnover_rate",
    "def_reb_rate"
]

for col in final_columns:
    if col not in df.columns:
        df[col] = 0.0

df = df[final_columns]
display(df.head())

,team,r_3pt_radius,court_width,baseline_width,hoop_to_baseline,corner3_distance,corner3_availability,arc_length_inbounds,area_inside_arc,area_outside_arc,...,threepar,team_rim_rate,team_corner3_rate,off_reb_rate,pace,opp_3par_allowed,opp_rim_fg_allowed,switch_rate,turnover_rate,def_reb_rate
0,Cavaliers2016,23.75,50,50,4,22,1.0,74.612826,886.027303,1263.972697,...,0.362,0.292,0.087,0.208,93.3,0.303,0.604,0.55,0.131,0.782
1,Warriors2016,23.75,50,50,4,22,1.0,74.612826,886.027303,1263.972697,...,0.416,0.285,0.095,0.229,99.3,0.287,0.582,0.72,0.149,0.789
2,Thunder2025,23.75,50,50,4,22,1.0,74.612826,886.027303,1263.972697,...,0.423,0.301,0.101,0.241,100.8,0.348,0.596,0.70,0.154,0.792
3,Pacers2025,23.75,50,50,4,22,1.0,74.612826,886.027303,1263.972697,...,0.398,0.287,0.094,0.226,101.5,0.356,0.603,0.62,0.139,0.774
4,Cavaliers2016,24.50,50,50,4,22,1.0,76.969020,942.870495,1207.129505,...,0.362,0.292,0.087,0.208,93.3,0.303,0.604,0.55,0.131,0.782


In [20]:
df.to_csv("final_nn_input.csv", index=False)
print("Saved final_nn_input.csv")

Saved final_nn_input.csv


In [22]:
import pandas as pd
import numpy as np

# -----------------------------------
# TEAM STATS TABLE
# -----------------------------------

team_stats_df = pd.DataFrame({
    "team": ["Cavaliers2016", "Warriors2016", "Thunder2025", "Pacers2025"],

    "twopa_per_game": [54.4, 55.7, 53.9, 53.5],
    "threepa_per_game": [29.6, 31.6, 38.8, 35.8],
    "corner3_pa": [7.3, 8.3, 9.4, 8.4],
    "above_break3_pa": [22.3, 23.3, 29.4, 27.4],
    "rim_attempt_share": [0.292, 0.285, 0.301, 0.287],
    "midrange_share": [0.356, 0.353, 0.280, 0.312],
    "free_throw_rate": [0.258, 0.250, 0.220, 0.242],

    "eppg": [0.0, 0.0, 0.0, 0.0],
    "epps": [0.0, 0.0, 0.0, 0.0],
    "efg_expected": [0.0, 0.0, 0.0, 0.0],

    "threepar": [0.362, 0.416, 0.423, 0.398],
    "team_rim_rate": [0.292, 0.285, 0.301, 0.287],
    "team_corner3_rate": [0.087, 0.095, 0.101, 0.094],
    "off_reb_rate": [0.208, 0.229, 0.241, 0.226],
    "pace": [93.3, 99.3, 100.8, 101.5],
    "opp_3par_allowed": [0.303, 0.287, 0.348, 0.356],
    "opp_rim_fg_allowed": [0.604, 0.582, 0.596, 0.603],
    "switch_rate": [0.55, 0.72, 0.70, 0.62],
    "turnover_rate": [0.131, 0.149, 0.154, 0.139],
    "def_reb_rate": [0.782, 0.789, 0.792, 0.774]
})

# -----------------------------------
# MERGE TEAM STATS ONTO GEOMETRY GRID
# -----------------------------------

final_df = df.merge(team_stats_df, on="team", how="left")

# -----------------------------------
# ORDER COLUMNS
# -----------------------------------

final_columns = [
    "team",
    "r_3pt_radius",
    "court_width",
    "baseline_width",
    "hoop_to_baseline",
    "corner3_distance",
    "corner3_availability",
    "arc_length_inbounds",
    "area_inside_arc",
    "area_outside_arc",
    "twopa_per_game",
    "threepa_per_game",
    "corner3_pa",
    "above_break3_pa",
    "rim_attempt_share",
    "midrange_share",
    "free_throw_rate",
    "eppg",
    "epps",
    "efg_expected",
    "threepar",
    "team_rim_rate",
    "team_corner3_rate",
    "off_reb_rate",
    "pace",
    "opp_3par_allowed",
    "opp_rim_fg_allowed",
    "switch_rate",
    "turnover_rate",
    "def_reb_rate"
]

final_df = final_df[final_columns]

print("Final dataset shape:", final_df.shape)
display(final_df.head())

final_df.to_csv("final_nn_input_full_grid.csv", index=False)
print("Saved final_nn_input_full_grid.csv")

Final dataset shape: (1092, 30)


,team,r_3pt_radius,court_width,baseline_width,hoop_to_baseline,corner3_distance,corner3_availability,arc_length_inbounds,area_inside_arc,area_outside_arc,...,threepar,team_rim_rate,team_corner3_rate,off_reb_rate,pace,opp_3par_allowed,opp_rim_fg_allowed,switch_rate,turnover_rate,def_reb_rate
0,Cavaliers2016,23.0,50.00,50.00,4.0,22,1.0,72.256631,830.951257,1319.048743,...,0.362,0.292,0.087,0.208,93.3,0.303,0.604,0.55,0.131,0.782
1,Cavaliers2016,23.0,50.25,50.25,4.0,22,1.0,72.256631,830.951257,1329.798743,...,0.362,0.292,0.087,0.208,93.3,0.303,0.604,0.55,0.131,0.782
2,Cavaliers2016,23.0,50.50,50.50,4.0,22,1.0,72.256631,830.951257,1340.548743,...,0.362,0.292,0.087,0.208,93.3,0.303,0.604,0.55,0.131,0.782
3,Cavaliers2016,23.0,50.75,50.75,4.0,22,1.0,72.256631,830.951257,1351.298743,...,0.362,0.292,0.087,0.208,93.3,0.303,0.604,0.55,0.131,0.782
4,Cavaliers2016,23.0,51.00,51.00,4.0,22,1.0,72.256631,830.951257,1362.048743,...,0.362,0.292,0.087,0.208,93.3,0.303,0.604,0.55,0.131,0.782


Saved final_nn_input_full_grid.csv
